# SMC Stochastic Volatility Project

**Course:** Sequential Monte Carlo (`particles`) — Prof. Nicolas Chopin  
**Presentation Date:** December 17  

---

## 1. Data & Base Setup (Gabriel)

### 1.1 Mathematical Formulation of the Base State-Space Model
Following Chopin & Papaspiliopoulos (2020, Section 2.4.3 & Chapter 4), we model daily asset log-returns $Y_t$ driven by an unobserved auto-regressive log-volatility process $X_t$:

$$\begin{aligned}
X_0 &\sim \mathcal{N}\!\left(\mu, \frac{\sigma^2}{1 - \phi^2}\right), \quad |\phi| < 1, \; \sigma > 0 \\
X_t \mid X_{t-1} = x_{t-1} &\sim \mathcal{N}\!\left(\mu + \phi(x_{t-1} - \mu), \, \sigma^2\right), \quad t \ge 1 \\
Y_t \mid X_t = x_t &\sim \mathcal{N}\!\left(0, \, \exp(x_t)\right), \quad t \ge 0
\end{aligned}$$

The base `StochVol` (`StochasticVolatility`) class in `src/base_model.py` implements `PX0`, `PX`, and `PY` in `particles`.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on sys.path when running from notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.base_model import (
    StochVol,
    TRUE_SIM_PARAMS,
    load_simulated_dataset,
    load_sp500_dataset,
    run_baseline_bootstrap,
    run_parallel_smc,
    plot_sandbox_validation,
    plot_sp500_overview_and_baseline,
)

### 1.2 Milestone 2: Synthetic Sandbox Dataset & Baseline Bootstrap Filter Validation
We load the simulated sandbox dataset (`data/processed/simulated_sv_data.csv`, generated with ground-truth parameters $\mu = -1.0, \phi = 0.95, \sigma = 0.2, T = 500$) and verify that the baseline Bootstrap particle filter tracks the hidden volatility state $X_t$.

In [ ]:
sim_data = load_simulated_dataset()
print("Ground-truth sandbox parameters:", TRUE_SIM_PARAMS)
display(sim_data["df"].head())

boot_sim = run_baseline_bootstrap(
    y_data=sim_data["y_obs"],
    model=StochVol(**TRUE_SIM_PARAMS),
    N=1000,
    x_true=sim_data["x_true"],
)
par_sim = run_parallel_smc(
    y_data=sim_data["y_obs"],
    model=StochVol(**TRUE_SIM_PARAMS),
    N=1000,
    nruns=8,
    nprocs=0,
    x_true=sim_data["x_true"],
)
print(
    f"Single-run RMSE(X): {boot_sim['metrics']['rmse_x']:.4f} | "
    f"MSE(X): {boot_sim['metrics']['mse_x']:.4f} | "
    f"Mean ESS: {boot_sim['metrics']['mean_ess']:.1f}"
)
print(
    f"Parallel 8-run RMSE(X): {par_sim['rmse_mean']:.4f} +/- {par_sim['rmse_std']:.4f} | "
    f"logL: {par_sim['logL_mean']:.2f} +/- {par_sim['logL_std']:.2f}"
)

fig_sim = plot_sandbox_validation(sim_data, boot_sim, save_path=None)
plt.show()

### 1.3 Milestone 3: Cleaned S&P 500 Log-Returns Dataset & Parallel Baseline Run
We load the cleaned S&P 500 daily percentage log-returns (`data/processed/sp500_log_returns.csv`, covering 2019-07-01 to 2021-06-30 including the Feb–Apr 2020 COVID-19 market crash) and verify the baseline Bootstrap particle filter on real market data.

In [ ]:
sp500_data = load_sp500_dataset()
display(
    sp500_data["df"][
        ["Date", "Close", "log_return", "pct_log_return", "demeaned_pct_log_return", "rolling_vol_20d"]
    ].head()
)

boot_sp500 = run_baseline_bootstrap(
    y_data=sp500_data["y_obs"],
    model=StochVol(mu=-0.8, rho=0.97, sigma=0.25),
    N=1000,
)
fig_sp = plot_sp500_overview_and_baseline(sp500_data, boot_sp500, save_path=None)
plt.show()

---

## 2. Live Tracking / Filtering (Role 2 — Filtering Lead)

*Section for Role 2: Taylor expansion math for Guided & Auxiliary Particle Filters (APF), ESS/MSE comparison table, and volatility tracking plots.*

In [ ]:
# Phase 2: Live Tracking / Filtering (Bootstrap, Guided, APF)


---

## 3. Hindsight Smoothing (Role 3 — Smoothing Lead)

*Section for Role 3: Forward Filtering Backward Sampling (FFBS) $O(N^2)$ vs. fast variants and two-filter smoothing on sandbox and S&P 500 data.*

In [ ]:
# Phase 3: Hindsight Smoothing (FFBS)


---

## 4. Parameter Estimation (Role 4 — Parameter Inference Lead)

*Section for Role 4: Prior distributions for $\theta = (\mu, \phi, \sigma^2)$ and PMCMC / $\text{SMC}^2$ posterior plots.*

In [ ]:
# Phase 4: Parameter Estimation (PMCMC, SMC^2)
